# trajopt-jax: a low-thrust transfer between planets, the launch date solved for

A spacecraft with a solar-electric engine leaves one planet on a launch date it chooses and meets another on a fixed arrival date, using as little propellant as it can. Any pair of planets from Mercury to Neptune; a rendezvous (match the target's position and velocity) or a flyby (its position only).

Source: [`examples/transfer.py`](https://github.com/thearn/trajopt-jax/blob/main/examples/transfer.py)

## The problem

**States** $\mathbf{s} = (\mathbf{r}, \mathbf{v}, m, t, c)$: heliocentric position $\mathbf{r}$ (AU) and velocity $\mathbf{v}$ (AU/yr), mass $m$ as a fraction of the starting mass, calendar time $t$ (years from the arrival date) and the cost $c$.

**Control** $\mathbf{u} \in \mathbb{R}^3$: the thrust as a fraction of the engine's maximum, a vector, $\lVert \mathbf{u} \rVert \le 1$.

**Parameters**

| symbol | meaning | default |
|---|---|---|
| $\mu_\odot = 4\pi^2\ \mathrm{AU^3/yr^2}$ | the Sun's gravitational parameter (Kepler's third law in these units) | |
| $T_{\max}$ | the engine's thrust | 0.5 N |
| $I_{sp}$ | its specific impulse | 3000 s |
| $m_0$ | the spacecraft's starting mass | 1500 kg |
| $g_0$ | standard gravity | 9.80665 m/s² |
| $v_{\infty,\max}$ | the launch vehicle's hyperbolic excess speed ($C_3 = v_\infty^2$) | 3 km/s |
| $t_a$ | the arrival date | 2033-07-08 |

**Dynamics**: two-body motion about the Sun with the engine's acceleration, the mass falling with the propellant's flow,
$$
\dot{\mathbf{r}} = \mathbf{v}, \qquad
\dot{\mathbf{v}} = -\frac{\mu_\odot}{\lVert\mathbf{r}\rVert^3}\,\mathbf{r} + \frac{T_{\max}}{m_0\, m}\,\mathbf{u}, \qquad
\dot m = -\frac{T_{\max}}{I_{sp}\, g_0\, m_0}\,\lVert\mathbf{u}\rVert, \qquad
\dot t = 1 .
$$

**Objective**: the least propellant, $m_0\,(1 - m(t_a))$. The cost's rate blends propellant with an energy-like term through a weight $w$,
$$
\dot c = \frac{T_{\max}}{I_{sp}\, g_0\, m_0} \left( w\, \lVert\mathbf{u}\rVert + (1 - w)\, \lVert\mathbf{u}\rVert^2 \right), \qquad \min J = c(t_a),
$$
solved for $w = 0, 0.5, 0.9, 1$ in turn, each solve seeded by the last (a homotopy). At $w = 1$ the optimum flies full thrust or none (bang-bang), and $\lVert\mathbf{u}\rVert$ has a corner at zero thrust that makes a solve from a rough seed creep; the $w = 0$ problem is smooth and finds the shape of the transfer first.

**Departure** (a free start, held by constraints): on a launch date $t_0$ in the window, at the origin planet, leaving at most at the launch vehicle's speed, with all the starting mass,
$$
\mathbf{r}(t_0) = \mathbf{r}_o(t_0), \qquad \lVert \mathbf{v}(t_0) - \mathbf{v}_o(t_0) \rVert \le v_{\infty,\max}, \qquad t_a - T_{\mathrm{hi}} \le t_0 \le t_a - T_{\mathrm{lo}}, \qquad m(t_0) = 1 .
$$

**Arrival** on the fixed date, at the target planet:
$$
t_f = t_a, \qquad \mathbf{r}(t_a) = \mathbf{r}_T(t_a), \qquad \mathbf{v}(t_a) = \mathbf{v}_T(t_a) \ \text{(rendezvous only)} .
$$
The flight time $t_a - t_0$ is free; its window defaults to 0.5 to 1.6 times the Hohmann transfer's.

**Ephemerides**: $\mathbf{r}_o, \mathbf{v}_o$ (the origin, a function of the free launch date) are Chebyshev series fitted once to astropy's built-in ephemeris over the launch window and evaluated in JAX, so the departure constraint is smooth and differentiable in $t_0$. The target is needed at one fixed date, so it is a single ephemeris lookup.

**Scope**: patched conics. The flight starts at the origin's sphere of influence with a hyperbolic excess speed (no escape spiral) and ends there at the target (no capture); no other planet's gravity acts. The usual first cut in mission design.

## How trajopt-jax transcribes it

One leg of $N$ stages (120 here: a few days each). Each stage's state is a cubic collocated at the Radau IIA points; the control is linear between knots. The departure is a *start event*: the position, velocity and launch date are left free at the first stage (`Stages.start_free`) and held by the constraints above (`start_events`), while the arrival is an end event on the leg. The interior-point solver handles the bounds and inequalities; its linear algebra is a Riccati recursion over the stages.

In [ ]:
try:  # on Colab: install trajopt-jax and astropy
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/thearn/trajopt-jax astropy scipy matplotlib
    !wget -q https://raw.githubusercontent.com/thearn/trajopt-jax/main/examples/transfer.py
except ImportError:
    import sys
    sys.path.insert(0, 'examples' if __import__('os').path.isdir('examples') else '.')

## Earth to Mars, arriving 2033-07-08

`solve()` in [`transfer.py`](https://github.com/thearn/trajopt-jax/blob/main/examples/transfer.py) fits the ephemeris, builds the problem, runs the homotopy and returns the launch date, flight time, propellant and departure speed, plus a check: the solution re-flown by SciPy's integrator from its own start with its own controls, and how far that lands from the transcription's end.

In [ ]:
import transfer as tr

out = tr.solve('earth', 'mars', '2033-07-08')
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in out.items() if k not in ('X', 'U', 'prob')}

The trajectory in the ecliptic's neighbourhood (the equatorial frame's x-y plane), coloured by throttle, with the planets' orbits; and the thrust history.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

prob = out['prob']
s = out['X'][:, :prob.ns] * prob.ref
u = out['X'][:, prob.ns:prob.ns + 3]
throttle = np.linalg.norm(u, axis=1)
days = (s[:, 7] - s[0, 7]) * 365.25

fig, (a0, a1) = plt.subplots(1, 2, figsize=(13, 5.5))
for body, colour in (('earth', 'tab:blue'), ('mars', 'tab:red')):
    from astropy.time import Time, TimeDelta
    dates = Time(out['arrival'], scale='tdb') + TimeDelta(np.linspace(-700, 0, 300), format='jd')
    pos, _ = tr.heliocentric(body, dates.iso)
    a0.plot(pos[:, 0], pos[:, 1], color=colour, lw=0.8, label=body)
sc = a0.scatter(s[:, 0], s[:, 1], c=throttle, cmap='magma', vmin=0, vmax=1, s=8)
a0.plot(0, 0, 'o', color='gold', ms=10)
a0.set_aspect('equal'); a0.set_xlabel('x (AU)'); a0.set_ylabel('y (AU)'); a0.legend()
fig.colorbar(sc, ax=a0, label='throttle')
a1.plot(days, throttle)
a1.set_xlabel('days from launch'); a1.set_ylabel('throttle |u|'); a1.set_ylim(-0.05, 1.05)
fig.suptitle(f"launch {out['launch']}, {out['flight_days']:.0f} days, {out['propellant_kg']:.0f} kg of propellant");

## Other worlds

The same call takes any pair from Mercury to Neptune. To the outer planets an electric engine alone (no gravity assists) rarely has the thrust for a rendezvous on a reasonable date; a flyby is how such missions are first sized. The window and stage count may need widening for long flights: `window=(lo, hi)` in years, `segments`.

In [ ]:
venus = tr.solve('earth', 'venus', '2033-07-08')
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in venus.items() if k not in ('X', 'U', 'prob')}